# 05 · SinDDM on Colab: port check, training speed, resume (Phase 2)

**T4 GPU runtime.** About 25–35 min.

SinDDM pins Python 3.8 / torch 1.13, which Colab cannot install, so we run our port:
`patches/sinddm_port/` replaces 3 files (see `CHANGES.diff`):
- a device option;
- resumable, time-budgeted training;
- safe weights-only checkpoint loading;
- a scikit-image API fix.

This notebook:
1. samples the 3 official pretrained models;
2. measures **seconds per 1 000 training steps** at the native size and at a 128-px short side;
3. tests stop-and-resume.

In [ ]:
# ==== SETTINGS (edit once) ====================================================================
CODE_SOURCE   = "github"                                      # "github" or "drive"
GITHUB_REPO   = "https://github.com/rishabh-vr/CS787-Single-Image-Diffusion-Editor.git"
GITHUB_BRANCH = "main"
REPO_SUBDIR   = ""                                            # project files are at the repo root
DRIVE_CODE    = "/content/drive/MyDrive/CS787/code/Project"   # used when CODE_SOURCE == "drive"
DRIVE_ROOT    = "/content/drive/MyDrive/CS787"                # outputs, checkpoints, tables, env logs
SID_COMMIT    = "85381e3c6f29fc07144da854d86be07150404716"     # official code, audited commit
SINDDM_COMMIT = "e7107d9fd198928f26e278f99ec428e261726a45"  # SinDDM, audited commit
# If the GitHub repo is private: add a Colab secret named GH_TOKEN (key icon in the left bar)
# holding a fine-grained, read-only token for that repo.

NOTEBOOK = "05_sinddm_port"

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU (then rerun)."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| bf16 native:", torch.cuda.get_device_capability(0) >= (8, 0))

In [ ]:
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
for sub in ("outputs", "checkpoints", "tables", "env_logs", "data"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

PROJECT = "/content/project"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
if CODE_SOURCE == "github":
    url = GITHUB_REPO
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")
    except Exception:
        pass  # public repo: no token needed
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", GITHUB_BRANCH, url, "/content/repo"], check=True)
    shutil.copytree(os.path.join("/content/repo", REPO_SUBDIR), PROJECT,
                    ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
else:
    shutil.copytree(DRIVE_CODE, PROJECT, ignore=shutil.ignore_patterns(".venv", "external", "outputs*"))
print("project code at", PROJECT, "->", sorted(os.listdir(PROJECT)))

In [ ]:
SINDDM = "/content/SinDDM"
if not os.path.exists(SINDDM):
    subprocess.run(["git", "clone", "-q", "https://github.com/fallenshock/SinDDM", SINDDM], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", SINDDM, "checkout", "-q", SINDDM_COMMIT], check=True)
shutil.copytree(f"{PROJECT}/patches/sinddm_port", SINDDM, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("CHANGES.diff"))
!pip install -q einops ftfy
print(subprocess.run(["git", "-C", SINDDM, "status", "--short"], capture_output=True, text=True).stdout)

## 1 · Sample the 3 official pretrained models (validates the port)

In [ ]:
%cd {SINDDM}
for scope in ("forest", "night_sky", "pyramids"):
    cmd = (f"python main.py --scope {scope} --mode sample --dataset_folder ./datasets/{scope}/ "
           f"--image_name {scope}.png --results_folder ./results/ --load_milestone 12 "
           f"--sample_batch_size 4 --seed 0 2>&1 | grep -v -i warn | tail -3")
    !{cmd}
# If loading fails with "Unsupported global", the official checkpoint holds another type:
# add it to the allow-list in patches/sinddm_port/SinDDM/trainer.py (load) and rerun.

In [ ]:
import glob
from IPython.display import Image as IPImage, display
for scope in ("forest", "night_sky", "pyramids"):
    grid = sorted(glob.glob(f"{SINDDM}/results/{scope}/final_samples/*_out_s*"))
    if grid:
        print(scope); display(IPImage(grid[-1], width=600))

## 2 · Training speed (seconds per 1 000 steps)

In [ ]:
import time, json
from PIL import Image
def make_input(name, short_side=None):
    d = f"/content/sinddm_bench/{name}/"; os.makedirs(d, exist_ok=True)
    im = Image.open(f"{SINDDM}/datasets/balloons/balloons.png").convert("RGB")
    if short_side:
        w, h = im.size; s = short_side / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.LANCZOS)
    im.save(d + f"{name}.png"); return d, im.size

results = {}
for name, short in (("bench_native", None), ("bench_128", 128)):
    folder, size = make_input(name, short)
    cmd = (f"python main.py --scope {name} --mode train --dataset_folder {folder} --image_name {name}.png "
           f"--results_folder /content/sinddm_bench/results/ --train_num_steps 1000 --save_and_sample_every 1000000 "
           f"--time_budget_min 20 --checkpoint_every_min 100 --sample_batch_size 1 --seed 0 "
           f"2>&1 | grep -E 'step:|port|completed' | tail -3")
    t0 = time.time()
    !{cmd}
    results[name] = {"input_wh": size, "seconds_per_1000_steps_incl_final_sampling": round(time.time() - t0, 1)}
print(json.dumps(results, indent=2))
full = 120_000
for k, v in results.items():
    print(f"{k}: full 120k-step schedule would take ~{v['seconds_per_1000_steps_incl_final_sampling'] * full / 1000 / 3600:.1f} h on this GPU (upper bound)")
json.dump(results, open(f"{DRIVE_ROOT}/tables/phase2_sinddm_timing.json", "w"), indent=2)

## 3 · Stop-and-resume test (checkpoints on Drive)

In [ ]:
ck = f"{DRIVE_ROOT}/checkpoints/sinddm_resume_test/"
folder, _ = make_input("resume_test", 128)
cmd = (f"python main.py --scope resume_test --mode train --dataset_folder {folder} --image_name resume_test.png "
       f"--results_folder {ck} --train_num_steps 400 --save_and_sample_every 1000000 --resume "
       f"--time_budget_min 0.5 --checkpoint_every_min 0.25 --sample_batch_size 1 --seed 0 "
       f"2>&1 | grep -E 'port|completed' | tail -3")
for attempt in (1, 2):
    print(f"--- run {attempt}")
    !{cmd}
print("expected: run 1 stops on the time budget; run 2 says 'resumed from step N' with N > 0")

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H%M")
!pip freeze > {DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt
print("saved", f"{DRIVE_ROOT}/env_logs/{stamp}_{NOTEBOOK}_freeze.txt")